# 🟩🟨⬜Wordle: Human vs Greedy Hill Climbing

This notebook implements:

1. A Wordle game engine.
2. An interactive manual game loop for a human player.
3. A greedy hill-climbing solver for comparison.


In [ ]:
from pathlib import Path
from collections import Counter
import random


def load_word_list(source):
    words = [
        line.strip().lower()
        for line in source.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    words = [w for w in words if len(w) == 5 and w.isalpha()]

    if not words:
        raise ValueError("Word list is empty after filtering for 5-letter alphabetic words.")


    return words, source


WORDS, WORD_SOURCE = load_word_list(Path("words.txt"))
ALLOWED_WORDS, ALLOWED_SOURCE = load_word_list(Path("allowed.txt"))
MAX_TURNS = 6

print(f"Loaded {len(WORDS)} words from {WORD_SOURCE.name}")

## Wordle Game Engine

The feedback engine returns one status per letter:

- `G` for green
- `Y` for yellow
- `B` for grey

The algorithm handles duplicate letters correctly by assigning greens first, then yellows from remaining counts.

In [ ]:
EMOJI_MAP = {"G": "🟩", "Y": "🟨", "B": "⬜"}


def wordle_feedback(secret, guess):
    """Return tuple feedback using G/Y/B symbols with duplicate-letter correctness."""
    secret = secret.lower()
    guess = guess.lower()

    feedback = ["B"] * 5
    remaining = Counter()

    # Pass 1: mark greens and count remaining secret letters.
    for i, (s, g) in enumerate(zip(secret, guess)): # (0, ('a', 'a')), (1, ('p', 'l'))
        if g == s:
            feedback[i] = "G"
        else:
            remaining[s] += 1

    # Pass 2: mark yellows using remaining counts.
    for i, g in enumerate(guess):
        if feedback[i] == "G":
            continue
        if remaining[g] > 0:
            feedback[i] = "Y"
            remaining[g] -= 1

    return tuple(feedback)


def feedback_to_emoji(feedback):
    return "".join(EMOJI_MAP[c] for c in feedback)


def format_guess_line(guess, feedback):
    return f"{guess.upper()} {feedback_to_emoji(feedback)}"


def validate_guess(guess, words):
    g = guess.strip().lower()
    if len(g) != 5 or not g.isalpha():
        return False, "Guess must be exactly 5 letters."
    if g not in words:
        return False, "Guess is not in the word list."
    return True, g

In [ ]:
# Quick Counter demo for duplicate-letter behavior.
example_secret = "apple"
example_guess = "allee"
example_feedback = wordle_feedback(example_secret, example_guess)
print(f"Secret: {example_secret.upper()} | Guess: {example_guess.upper()}")
print(format_guess_line(example_guess, example_feedback))
print("Legend: G=🟩, Y=🟨, B=⬜")

## Why Counter Is Used (and How Colors Are Assigned)

`Counter` from `collections` counts how many times each letter appears.

In Wordle feedback with duplicates, we must avoid over-counting yellow letters:

1. Mark all greens first (`G` / 🟩).
2. Count only the remaining unmatched letters in the secret using `Counter`.
3. For each non-green letter in the guess:
   - If that letter still has count > 0, mark yellow (`Y` / 🟨) and decrease the count.
   - Otherwise mark grey (`B` / ⬜).

That is exactly why `Counter` is the safest way to make Wordle colors correct.

## Manual Mode

Run the next cell to play Wordle yourself.

For each guess, the notebook prints:

`GUESS` + emoji feedback, for example `POWER 🟩⬜🟨⬜🟩`

In [ ]:
def play_wordle_manual(secret, allowed_words, max_turns=MAX_TURNS):
    """Interactive Wordle loop for a human player."""
    history = []

    print("\nWordle manual mode started.")
    print(f"You have {max_turns} turns. Enter 5-letter guesses from the word list.")

    for turn in range(1, max_turns + 1):
        while True:
            raw = input(f"Turn {turn}/{max_turns} guess: ")
            ok, value = validate_guess(raw, allowed_words)
            if ok:
                guess = value
                break
            print(f"Invalid guess: {value}")

        feedback = wordle_feedback(secret, guess)
        history.append((guess, feedback))
        print(format_guess_line(guess, feedback))

        if guess == secret:
            print(f"Solved in {turn} turns.")
            return {
                "solved": True,
                "turns": turn,
                "history": history,
                "secret": secret,
            }

    print("Not solved within turn limit.")
    return {
        "solved": False,
        "turns": max_turns,
        "history": history,
        "secret": secret,
    }

## Greedy Hill Climbing Solver

This solver follows your required logic exactly:

- Step A: start with full candidate list.
- Step B: recalculate **positional** letter frequencies from candidates only (one Counter per position).
- Step C: score each candidate by summing the positional frequency of each letter at its exact position.
- Step D: pick highest score, alphabetical tie-break.
- Step E: prune candidates using observed feedback.
- Step F: repeat until solved.

In [ ]:
def compute_positional_frequencies(candidates):
    """Step B: positional frequency table — one Counter per position (0-4)."""
    pos_freq = [Counter() for _ in range(5)]
    for word in candidates:
        for i, ch in enumerate(word):
            pos_freq[i][ch] += 1
    return pos_freq


def score_candidate(word, pos_freq):
    """Step C: sum positional frequencies — rewards letters in their most common positions."""
    return sum(pos_freq[i][ch] for i, ch in enumerate(word))


def choose_greedy_guess(candidates):
    """Step D: max score; alphabetical tie-break."""
    pos_freq = compute_positional_frequencies(candidates)

    # Sort by (-score, word) so ties are broken alphabetically.
    ranked = sorted(
        candidates,
        #              v negative because sorted() does ascending order by default, and if we reverse = true, it would reverse the word order as well
        key=lambda w: (-score_candidate(w, pos_freq), w),
    )
    best = ranked[0]
    best_score = score_candidate(best, pos_freq)
    return best, best_score, pos_freq


def find_diagnostic_guess(tied_candidates, allowed_words):
    """Find a word that distinguishes among tied candidates.

    A distinguishing letter is one that appears in some but not all tied
    candidates — guessing it reveals which sub-group the secret belongs to.
    We scan allowed_words for the word that covers the most such letters.
    """
    all_letters  = set(ch for w in tied_candidates for ch in w)
    # Letters present in every tied candidate give no new information.
    universal    = {ch for ch in all_letters if all(ch in w for w in tied_candidates)}
    distinguishing = all_letters - universal

    # Pick the allowed word that tests the most distinguishing letters; alphabetical tie-break.
    ranked = sorted(
        allowed_words,
        key=lambda w: (-len(set(w) & distinguishing), w),
    )
    return ranked[0], distinguishing


def prune_candidates(candidates, guess, observed_feedback):
    """Step E: keep words whose feedback vs guess matches observed feedback."""
    return [
        candidate_word for candidate_word in candidates if wordle_feedback(candidate_word, guess) == observed_feedback
    ]


def solve_wordle_greedy_hill(secret, words, max_turns=MAX_TURNS, printing=True):
    """Greedy hill climbing Wordle solver implementing Steps A-F."""
    # Step A: Start with the full word list as candidates.
    candidates = list(words)
    history = []

    if printing:
        print(f"Step A: start with {len(candidates)} candidates")

    for turn in range(1, max_turns + 1):
        # Steps B, C, D happen inside this guess selection call.
        guess, guess_score, pos_freq = choose_greedy_guess(candidates)

        # Count candidates that share the top score (the tied group).
        tied = [w for w in candidates if score_candidate(w, pos_freq) == guess_score]
        num_tied       = len(tied)
        remaining_turns = max_turns - turn  # turns available after this one

        # Diagnostic override: when tied candidates exceed remaining turns the
        # greedy approach cannot guarantee a solve, so look for a better probe.
        using_diagnostic = False
        if num_tied > remaining_turns and num_tied > 1:
            diag_word, distinguishing = find_diagnostic_guess(tied, ALLOWED_WORDS)
            greedy_dist_count = len(set(guess) & distinguishing)
            diag_dist_count   = len(set(diag_word) & distinguishing)
            # Only override if the diagnostic word is strictly more informative.
            if diag_dist_count > greedy_dist_count:
                guess = diag_word
                using_diagnostic = True

        feedback = wordle_feedback(secret, guess)
        history.append((guess, feedback))

        if printing:
            # Combine positional counters into one for display purposes.
            combined_freq = Counter()
            for pf in pos_freq:
                combined_freq.update(pf)
            top_letters = sorted(combined_freq.items(), key=lambda kv: (-kv[1], kv[0]))[:5]
            top_letters_text = ", ".join(f"{ch}:{count}" for ch, count in top_letters)
            print(f"\nTurn {turn}")
            print(f"Step B: recomputed positional frequencies from {len(candidates)} candidates")
            print(f"Step C: score({guess.upper()}) = {guess_score} (positional)")
            print(f"Step D: chose best guess with alphabetical tie-break")
            if using_diagnostic:
                print(f"  [Diagnostic override] {num_tied} tied candidates, "
                      f"{remaining_turns} turns left — greedy cannot guarantee solve.")
                print(f"  Distinguishing letters: {''.join(sorted(distinguishing))}")
                print(f"  Diagnostic guess: {guess.upper()} "
                      f"(covers {len(set(guess) & distinguishing)} distinguishing letters)")
            print(f"Top letters this turn: {top_letters_text}")
            print(format_guess_line(guess, feedback))
            print("Colors: 🟩=green, 🟨=yellow, ⬜=grey")

        if guess == secret:
            if printing:
                print(f"Solved in {turn} turns.")
            return {
                "solved": True,
                "turns": turn,
                "history": history,
                "secret": secret,
            }

        # Step E: prune candidates, then Step F repeats loop.
        before = len(candidates)
        candidates = prune_candidates(candidates, guess, feedback)

        if printing:
            print(f"Step E: pruned candidates {before} -> {len(candidates)}")
            print("Step F: repeat with remaining candidates")

        if not candidates:
            break

    if printing:
        print("Greedy hill climber did not solve within turn limit.")
    return {
        "solved": False,
        "turns": max_turns,
        "history": history,
        "secret": secret,
    }

## Greedy Solver Benchmark (AI Generated for Presentation)

The cell below runs the greedy hill-climbing solver against **every word in `WORDS`** and reports how many were solved in each number of turns (1–6) or not solved at all.

This gives an objective measure of the algorithm's performance across the full answer space before you play against it interactively.

All loop variables are prefixed with `_` to avoid polluting the notebook namespace.

In [ ]:
from collections import Counter as _Counter

_total = len(WORDS)
_dist  = _Counter()

print(f"Benchmarking greedy solver on all {_total} words...\n")

for _i, _word in enumerate(WORDS, 1):
    _result = solve_wordle_greedy_hill(_word, WORDS, max_turns=MAX_TURNS, printing=False)
    _dist[_result["turns"] if _result["solved"] else 0] += 1
    if _i % 500 == 0 or _i == _total:
        print(f"  {_i}/{_total} done...")

_solved = sum(v for k, v in _dist.items() if k > 0)
_failed = _dist.get(0, 0)
_avg    = sum(k * v for k, v in _dist.items() if k > 0) / _solved if _solved else 0
_bar_w  = 30
_max_c  = max(_dist.values())

print(f"\n  Turns     Distribution{' ' * (_bar_w - 12)}  Count      %")
print("-" * (_bar_w + 26))
for _t in range(1, MAX_TURNS + 1):
    _n   = _dist.get(_t, 0)
    _pct = _n / _total * 100
    _bar = "#" * round(_n / _max_c * _bar_w)
    _lbl = f"{_t} turn" + ("s" if _t > 1 else " ")
    print(f"  {_lbl}  {_bar:<{_bar_w}}  {_n:4d} / {_total}  ({_pct:5.1f}%)")
if _failed:
    _pct = _failed / _total * 100
    _bar = "#" * round(_failed / _max_c * _bar_w)
    print(f"  Failed   {_bar:<{_bar_w}}  {_failed:4d} / {_total}  ({_pct:5.1f}%)")
print("-" * (_bar_w + 26))
print(f"\n  Solved : {_solved:4d} / {_total}  ({_solved / _total * 100:.1f}%)")
print(f"  Average: {_avg:.3f} turns  (solved games only)")

## Human vs AI Comparison

Run the next cell:

1. A random secret word is chosen from the same pool.
2. You play first in manual mode.
3. The greedy hill climber solves the exact same secret.
4. The notebook prints a turn comparison summary.

In [ ]:
# secret_word = random.choice(WORDS)
secret_word = "wight" # Make sure this is lowercase

print("New playthrough started.")
print("A random secret word was selected. Human plays first.\n")

human_result = play_wordle_manual(secret_word, ALLOWED_WORDS, max_turns=MAX_TURNS)

print("\nNow running greedy hill climber on the same secret...\n")
ai_result = solve_wordle_greedy_hill(secret_word, WORDS, max_turns=MAX_TURNS, printing=True)

print("\n===== Summary =====")
print(f"Secret word: {secret_word.upper()}")
print(f"Human solved: {human_result['solved']} | Turns: {human_result['turns']}")
print(f"AI solved:    {ai_result['solved']} | Turns: {ai_result['turns']}")

if human_result["solved"] and ai_result["solved"]:
    if human_result["turns"] < ai_result["turns"]:
        print("Winner: Human")
    elif human_result["turns"] > ai_result["turns"]:
        print("Winner: Greedy Hill Climber")
    else:
        print("Result: Tie")
elif human_result["solved"] and not ai_result["solved"]:
    print("Winner: Human (AI did not solve in time)")
elif ai_result["solved"] and not human_result["solved"]:
    print("Winner: Greedy Hill Climber (human did not solve in time)")
else:
    print("Result: Neither solved within turn limit")

---

## Interactive GUI (ipywidgets) (AI Generated for Presentation)

The GUI lives in **`wordle_gui.py`** (same directory as this notebook). The cells below just import it and launch it.

Gameplay:
- Type letters into the text box and press **Enter**, or click the on-screen keyboard.
- The **AI grid animates** turn-by-turn after your round ends.
- **Expand** toggles between normal (52 px) and large (68 px) tiles.
- **New Game** keeps the running score.

In [ ]:
from importlib import reload
import wordle_gui as _wg
reload(_wg)  # picks up edits to wordle_gui.py without restarting the kernel
from wordle_gui import make_game

In [ ]:
display(make_game(
    words=WORDS,
    allowed_words=ALLOWED_WORDS,
    max_turns=MAX_TURNS,
    wordle_feedback=wordle_feedback,
    feedback_to_emoji=feedback_to_emoji,
    choose_greedy_guess=choose_greedy_guess,
    prune_candidates=prune_candidates,
))